<a href="https://colab.research.google.com/github/claudiohenriquezberroeta-pucv/desafio_kaggle_5/blob/main/Desafio_6.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import kagglehub                #para importar datos del desafio
from google.colab import files  #para guardar datos en Drive (características y tokens)
from google.colab import drive  #para conectar tu cuenta de Google Drive directamente con tu entorno de Google Colab


from sklearn.cluster import KMeans    #para la clusterización en la función de tokenización
from sklearn.metrics import f1_score  #para utilizar la métrica de rendimiento f1_score
from sklearn.model_selection import train_test_split    #para la división de datos en conjuntos de entrenamiento y test
from sklearn.cluster import MiniBatchKMeans             #para la clusterización en la función de tokenización, más eficiente

import pandas as pd             #para la manipulación de datos
import numpy as np              #para la manipulación de arreglos
import os                       #para operar con los archivos y directorios
import random                   #para generar números aleatorios
import matplotlib.pyplot as plt #para graficar
import librosa                  #para el procesamiento de señales de audio
import librosa.display          #para visualizar de forma gráfica características de audio
import tensorflow as tf         #para trabajar con redes y tensores

#Elementos de tensorflow
from tensorflow.keras.utils import to_categorical       #para la converción de datos categórico (one hot)
from tensorflow.keras.layers import Input               #para definir la forma de los datos de entrada
from tensorflow.keras.layers import Dense               #para crear y definir la capa densa
from tensorflow.keras.layers import Dropout             #para definir la técnica de regularización dropout
from tensorflow.keras.layers import Embedding           #para la conversión de categorías en vectores densos de tamaño fijo
from tensorflow.keras.layers import GlobalMaxPooling1D  #para realizar la reducción (pooling) en datos temporales o secuenciales
from tensorflow.keras.models import Sequential          #para inicializar una pila lineal de capas
from tensorflow.keras.layers import Conv1D              #para importar una capa de convolución unidimensional en una Red Neuronal Convolucional
from tensorflow.keras.layers import MaxPooling1D        #para aplicar una capa de submuestreo (pooling) máximo sobre datos secuenciales
from tensorflow.keras.layers import Flatten             #para aplanar los datos de entrada
from tensorflow.keras.callbacks import EarlyStopping    #para definir criterios de detección temprana
from tensorflow.keras import regularizers               #para definir regulizadores L1 L2
from tensorflow.keras.preprocessing.sequence import pad_sequences   #para normalizar la longitud de listas de secuencias, transformándolas en un arreglo rectangular uniforme


In [4]:
# -*- coding: utf-8 -*-
"""
Módulo Integrado de Clasificación Automatizada de Estilos Musicales mediante
Segmentación en Memoria, Cuantización Vectorial Masiva y CNN 1D.

Optimizaciones: Vectorización completa mediante NumPy, persistencia MIR en Drive
y mitigación de redundancia I/O para conjuntos de entrenamiento y prueba.
"""
import warnings

# Supresión de alertas de ejecución para optimizar la salida limpia en consola
warnings.filterwarnings("ignore", category=FutureWarning, module="librosa")
warnings.filterwarnings("ignore", category=UserWarning, module="librosa")

In [6]:
# ==========================================
# 1. Configuración de Entorno y Descarga
# ==========================================
print("--- Paso 1: Configurando Kaggle y Google Drive ---")
files.upload() # Sube tu archivo kaggle.json aquí

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

path = kagglehub.competition_download('clasificacion-de-generos-musicales')
print("Ruta de los archivos de la competencia:", path)

drive.mount('/content/drive')

--- Paso 1: Configurando Kaggle y Google Drive ---


Saving kaggle.json to kaggle.json


100%|██████████| 6.34G/6.34G [01:12<00:00, 93.8MB/s]

Extracting files...


Ruta de los archivos de la competencia: /root/.cache/kagglehub/competitions/clasificacion-de-generos-musicales
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
# =====================================================================
# 1. CONEXIÓN AL ENTORNO DE ALMACENIMIENTO PERSISTENTE
# =====================================================================
print("--- Etapa 1: Conexión a Repositorios Persistentes ---")
drive.mount('/content/drive')

# Rutas globales de sincronización en Google Drive
ruta_salida_base = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Train/'
carpeta_mfccs_train = os.path.join(ruta_salida_base, 'mfccs/')

# Configuración de directorios de destino para el conjunto de pruebas en Drive
ruta_salida_test_drive = '/content/drive/MyDrive/Colab Notebooks/Kaggle_files/Test/'
carpeta_mfccs_test_drive = os.path.join(ruta_salida_test_drive, 'mfccs/')
carpeta_tokens_test_drive = os.path.join(ruta_salida_test_drive, 'tokens/tokens_mfcc/')

os.makedirs(carpeta_mfccs_test_drive, exist_ok=True)
os.makedirs(carpeta_tokens_test_drive, exist_ok=True)

# Descarga automatizada del corpus de datos desde el repositorio de la competencia
ruta_base_kaggle = kagglehub.competition_download('clasificacion-de-generos-musicales')
ruta_audio_test_crudo = os.path.join(ruta_base_kaggle, 'test')

--- Etapa 1: Conexión a Repositorios Persistentes ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [7]:
# =====================================================================
# 2. CARGA EFICIENTE Y SEGMENTACIÓN EN MEMORIA (IN-MEMORY CHUNKING - TRAIN)
# =====================================================================
print("\n--- Etapa 2: Segmentación y Alineación Vectorial en Memoria RAM ---")
df_train = pd.read_csv('train.csv')

X_mfcc_chunks_list = []
y_chunk_labels_list = []

# Parámetros invariantes de la ventana deslizante sobre marcos espectrales de MFCC
CHUNK_FRAMES = 430
HOP_FRAMES = int(CHUNK_FRAMES * 0.5) # Solapamiento del 50%

for index, row in df_train.iterrows():
    nombre_base = row['filename']
    id_cancion = os.path.splitext(nombre_base)[0]
    etiqueta = int(row['label'])

    # Estrategia de búsqueda flexible para prevenir fallos por sufijos de guardado previos
    ruta_opcion_1 = os.path.join(carpeta_mfccs_train, f"{id_cancion}_mfccs_entrenamiento.npy")
    ruta_opcion_2 = os.path.join(carpeta_mfccs_train, f"{id_cancion}.npy")

    if os.path.exists(ruta_opcion_1):
        ruta_completa_mfcc = ruta_opcion_1
    elif os.path.exists(ruta_opcion_2):
        ruta_completa_mfcc = ruta_opcion_2
    else:
        continue

    # Carga de la matriz espectral persistida. Dimensiones: (13, frames_totales)
    mfcc_cancion = np.load(ruta_completa_mfcc)

    # Transposición inmediata para consistencia en el eje temporal: (frames_totales, 13)
    if mfcc_cancion.shape[0] == 13:
        mfcc_cancion = mfcc_cancion.T

    frames_totales = mfcc_cancion.shape[0]

    # Segmentación por ventanas deslizantes directamente en memoria RAM
    for start in range(0, frames_totales - CHUNK_FRAMES + 1, HOP_FRAMES):
        chunk = mfcc_cancion[start:start + CHUNK_FRAMES, :]
        X_mfcc_chunks_list.append(chunk)
        y_chunk_labels_list.append(etiqueta)

print(f"Total de fragmentos acústicos estructurados en memoria (Train): {len(X_mfcc_chunks_list)}")


--- Etapa 2: Segmentación y Alineación Vectorial en Memoria RAM ---
Total de fragmentos acústicos estructurados en memoria (Train): 28770


In [9]:
# =====================================================================
# 3. ENTRENAMIENTO EXPRESO DEL CODEBOOK GLOBAL
# =====================================================================
print("\n--- Etapa 3: Cuantización Vectorial Masiva (Batch Quantization) ---")

np.random.seed(42)
indices_muestra = np.random.choice(len(X_mfcc_chunks_list), size=min(600, len(X_mfcc_chunks_list)), replace=False)
muestra_para_fit = [X_mfcc_chunks_list[idx] for idx in indices_muestra]

X_codebook_train = np.vstack(muestra_para_fit)

kmeans_global = MiniBatchKMeans(n_clusters=100, random_state=42, batch_size=8192)
kmeans_global.fit(X_codebook_train)
print("Estado: Codebook global consolidado de forma determinista.")


--- Etapa 3: Cuantización Vectorial Masiva (Batch Quantization) ---
Estado: Codebook global consolidado de forma determinista.


In [10]:
# =====================================================================
# 4. TOKENIZACIÓN VECTORIAL EN BLOQUE Y PREPARACIÓN DE TENSORES
# =====================================================================
X_all_chunks_stacked = np.vstack(X_mfcc_chunks_list)
all_tokens_predicted = kmeans_global.predict(X_all_chunks_stacked)
X_tokens_chunks_list = np.split(all_tokens_predicted, len(X_mfcc_chunks_list))

MAX_LEN = 300
X_final = pad_sequences(X_tokens_chunks_list, maxlen=MAX_LEN, padding='post', truncating='post')
y_final = np.array(y_chunk_labels_list)

X_temp, X_test, y_temp, y_test = train_test_split(
    X_final, y_final, test_size=0.1, stratify=y_final, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.2/0.9, stratify=y_temp, random_state=42
)

print(f"Estructuras del diseño experimental finalizadas -> Train: {X_train.shape}, Validation: {X_val.shape}, Test: {X_test.shape}")

Estructuras del diseño experimental finalizadas -> Train: (20139, 300), Validation: (5754, 300), Test: (2877, 300)


In [11]:
# =====================================================================
# 5. CONSTRUCCIÓN Y EVALUACIÓN DE TOPOLOGÍA CONVOLUCIONAL 1D
# =====================================================================
def calculate_f1(y_true, y_pred_probs):
    y_pred = np.argmax(y_pred_probs, axis=1)
    return f1_score(y_true, y_pred, average='macro')

def train_evaluate_cnn1d(activation, depth, neurons, learning_rate, optimizer_name='Adam',
                         batch_size=32, initializer='glorot_uniform', dropout_rate=0.2,
                         regularizer_type='l2', regularizer_lambda=0.0001, epochs=150, patience=25,
                         vocab_size=100, max_length=300):

    reg = regularizers.l2(regularizer_lambda) if regularizer_type == 'l2' else None

    model = Sequential()
    model.add(Input(shape=(max_length,), dtype='int32'))
    model.add(Embedding(input_dim=vocab_size, output_dim=64))

    model.add(Conv1D(filters=neurons, kernel_size=7, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg, padding='same'))
    model.add(MaxPooling1D(pool_size=3))
    if dropout_rate > 0: model.add(Dropout(dropout_rate))

    for _ in range(depth - 1):
        model.add(Conv1D(filters=neurons, kernel_size=5, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg, padding='same'))
        model.add(MaxPooling1D(pool_size=2))
        if dropout_rate > 0: model.add(Dropout(dropout_rate))

    model.add(GlobalMaxPooling1D())

    model.add(Dense(neurons, activation=activation, kernel_initializer=initializer, kernel_regularizer=reg))
    if dropout_rate > 0: model.add(Dropout(dropout_rate))
    model.add(Dense(8, activation='softmax'))

    if optimizer_name == 'Adam': optimizer = tf.keras.optimizers.Adam(learning_rate=learning_rate)
    elif optimizer_name == 'RMSprop': optimizer = tf.keras.optimizers.RMSprop(learning_rate=learning_rate)
    else: optimizer = tf.keras.optimizers.SGD(learning_rate=learning_rate)

    model.compile(optimizer=optimizer, loss='sparse_categorical_crossentropy', metrics=['accuracy'])

    # Corrección de la variable condicional para la detención temprana
    early_stop = EarlyStopping(monitor='val_loss', patience=patience, restore_best_weights=True)

    history = model.fit(
        X_train, y_train, validation_data=(X_val, y_val),
        epochs=epochs, batch_size=batch_size, callbacks=[early_stop], verbose=0
    )

    y_val_pred_probs = model.predict(X_val, verbose=0)
    y_test_pred_probs = model.predict(X_test, verbose=0)

    return history, calculate_f1(y_val, y_val_pred_probs), calculate_f1(y_test, y_test_pred_probs), model

In [ ]:
# =====================================================================
# 6. OPTIMIZACIÓN DE HIPERPARÁMETROS (RANDOM SEARCH)
# =====================================================================
print("\n--- Etapa 4: Ejecución del algoritmo de búsqueda aleatoria ---")
activations_list  = ['relu', 'tanh']
depths            = [1, 2, 3]
neurons_list      = [32, 64, 128]
learning_rates    = [0.001, 0.01]

fixed_batch_size  = 32
fixed_initializer = 'glorot_uniform'
fixed_optimizer   = 'Adam'
fixed_patience    = 20

num_trials = 20
results = []
best_f1 = -1
best_config = None

for i in range(num_trials):
    activation = random.choice(activations_list)
    depth = random.choice(depths)
    neurons = random.choice(neurons_list)
    lr = random.choice(learning_rates)

    print(f"\nEvaluación de Trial {i+1}/{num_trials} -> Topología CNN: act={activation}, depth={depth}, filtros={neurons}, lr={lr}")

    history, val_f1, test_f1, model = train_evaluate_cnn1d(
        activation=activation, depth=depth, neurons=neurons, learning_rate=lr,
        optimizer_name=fixed_optimizer, batch_size=fixed_batch_size,
        initializer=fixed_initializer, epochs=150, patience=fixed_patience
    )

    results.append({
        'activation': activation, 'depth': depth, 'neurons': neurons, 'learning_rate': lr, 'val_f1': val_f1, 'test_f1': test_f1
    })
    print(f"Métricas del Trial -> Validación F1: {val_f1:.4f} | Test F1: {test_f1:.4f}")

    if val_f1 > best_f1:
        best_f1 = val_f1
        best_config = results[-1]
        best_model_keras = model
        print("Notificación: Actualización del mejor estimador en memoria.")

print("\n===== ARQUITECTURA ÓPTIMA CONSOLIDADA =====")
print(best_config)

In [ ]:
# =====================================================================
# 6. OPTIMIZACIÓN DE HIPERPARÁMETROS (RANDOM SEARCH)
# =====================================================================
print("\n--- Etapa 4: Ejecución del algoritmo de búsqueda aleatoria ---")
activations_list  = ['relu', 'tanh']
depths            = [1, 2, 3]
neurons_list      = [32, 64, 128]
learning_rates    = [0.001, 0.01]

fixed_batch_size  = 32
fixed_initializer = 'glorot_uniform'
fixed_optimizer   = 'Adam'
fixed_patience    = 20

num_trials = 20
results = []
best_f1 = -1
best_config = None


activation = 'tanh'
depth = 2
neurons = 32
lr = 0.001


history, val_f1, test_f1, model = train_evaluate_cnn1d(
        activation=activation, depth=depth, neurons=neurons, learning_rate=lr,
        optimizer_name=fixed_optimizer, batch_size=fixed_batch_size,
        initializer=fixed_initializer, epochs=150, patience=fixed_patience
    )

print(f"Métricas del Trial -> Validación F1: {val_f1:.4f} | Test F1: {test_f1:.4f}")

print("\n===== ARQUITECTURA ÓPTIMA CONSOLIDADA =====")



--- Etapa 4: Ejecución del algoritmo de búsqueda aleatoria ---


In [ ]:
# =====================================================================
# 7. PIPELINE DE INFERENCIA SANEADO Y PERSISTENCIA DE METADATOS DE TEST
# =====================================================================
print("\n--- Etapa 5: Inferencia por ensamble de fragmentos (Votación Avanzada) ---")

# Inicialización y unificación homogénea del diccionario de traducción estructural de categorías
estilos_dict = np.load('dict.npy', allow_pickle=True).item()
inverso_estilos_dict_corregido = {}
for k, v in estilos_dict.items():
    try:
        inverso_estilos_dict_corregido[int(v)] = k
    except ValueError:
        inverso_estilos_dict_corregido[v] = k

# Indexación sistemática de las señales digitales .mp3 no etiquetadas descargadas de Kaggle
archivos_audio_test = [f for f in os.listdir(ruta_audio_test_crudo) if f.endswith('.mp3')]
print(f"Total de archivos de audio identificados en el repositorio Kaggle: {len(archivos_audio_test)}")

nombres_archivos_salida = []
predicciones_finales_texto = []

for archivo_audio_name in archivos_audio_test:
    ruta_completa_audio = os.path.join(ruta_audio_test_crudo, archivo_audio_name)
    id_cancion = os.path.splitext(archivo_audio_name)[0]

    try:
        # Carga analítica de la señal acústica sin re-muestreo
        y, sr = librosa.load(ruta_completa_audio, sr=None)

        # Extracción analítica de los 13 coeficientes cepstrales originales
        mfcc_nativo = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        # PERSISTENCIA EN GOOGLE DRIVE: Salvaguarda física de las características de Test
        nombre_guardado_mfcc_drive = f"{id_cancion}_mfccs_test.npy"
        np.save(os.path.join(carpeta_mfccs_test_drive, nombre_guardado_mfcc_drive), mfcc_nativo)

        # Transposición matemática para consistencia en el eje temporal: (frames_totales, 13)
        mfcc_transpuesto = mfcc_nativo.T
        frames_totales_test = mfcc_transpuesto.shape[0]

        chunks_tokens_cancion = []

        # Segmentación temporal por ventanas deslizantes homólogas en memoria RAM (In-Memory Chunking)
        for start in range(0, frames_totales_test - CHUNK_FRAMES + 1, HOP_FRAMES):
            chunk_test = mfcc_transpuesto[start:start + CHUNK_FRAMES, :]

            # Cuantización utilizando la proyección del Codebook global unificado del entrenamiento
            tokens_test_chunk = kmeans_global.predict(chunk_test)
            chunks_tokens_cancion.append(tokens_test_chunk)

        if chunks_tokens_cancion:
            # PERSISTENCIA EN GOOGLE DRIVE: Salvaguarda física de las secuencias de tokens de Test
            tokens_lineales_cancion = np.concatenate(chunks_tokens_cancion)
            nombre_guardado_token_drive = f"{id_cancion}_mfccs_test_tokens_mfccs_test.npy"
            np.save(os.path.join(carpeta_tokens_test_drive, nombre_guardado_token_drive), tokens_lineales_cancion)

            # Homogeneización dimensional mediante la adición de padding pos-secuencia
            X_chunks_pred = pad_sequences(chunks_tokens_cancion, maxlen=MAX_LEN, padding='post', truncating='post')

            # Inferencia probabilística a través de la topología óptima guardada
            probabilidades_chunks = best_model_keras.predict(X_chunks_pred, verbose=0)
            clases_numpy = np.argmax(probabilidades_chunks, axis=1)

            # Criterio de agregación estadística mediante la moda empírica (Votación por Mayoría)
            clase_ganadora_np = np.bincount(clases_numpy).argmax()
            clase_ganadora_python = int(clase_ganadora_np) # Saneamiento de tipos np.int64 para evitar KeyErrors

            nombres_archivos_salida.append(archivo_audio_name)

            # Mapeo categórico de resguardo tolerante contra strings y enteros
            if clase_ganadora_python in inverso_estilos_dict_corregido:
                predicciones_finales_texto.append(inverso_estilos_dict_corregido[clase_ganadora_python])
            else:
                clase_str = str(clase_ganadora_python)
                predicciones_finales_texto.append(inverso_estilos_dict_corregido.get(clase_str, clase_str))

    except Exception as e:
        print(f"Advertencia: Omisión de la pista {archivo_audio_name} debido a fallos estructurales. Error: {e}")

# Consolidación del DataFrame final y exportación a almacenamiento local
submission = pd.DataFrame({'filename': nombres_archivos_salida, 'label': predicciones_finales_texto})
submission.to_csv('submission_kaggle_mir.csv', index=False)

print("\nPipeline completado de forma canónica. Archivo exportado para evaluación externa.")
print(submission.head(10))#

--- Etapa 1: Conexión a Repositorios Persistentes ---
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

--- Etapa 2: Segmentación y Alineación Vectorial en Memoria RAM ---
Total de fragmentos acústicos estructurados en memoria (Train): 28770

--- Etapa 3: Cuantización Vectorial Masiva (Batch Quantization) ---
Estado: Codebook global consolidado de forma determinista.
Estructuras del diseño experimental finalizadas -> Train: (20139, 300), Validation: (5754, 300), Test: (2877, 300)

--- Etapa 4: Ejecución del algoritmo de búsqueda aleatoria ---

Evaluación de Trial 1/20 -> Topología CNN: act=tanh, depth=3, filtros=128, lr=0.01
Métricas del Trial -> Validación F1: 0.2063 | Test F1: 0.2056
Notificación: Actualización del mejor estimador en memoria.

Evaluación de Trial 2/20 -> Topología CNN: act=tanh, depth=3, filtros=32, lr=0.001
Métricas del Trial -> Validación F1: 0.4775 | Test F1: 0.4710
Notificación: Actua

In [5]:
# =====================================================================
# 7. PIPELINE DE INFERENCIA SANEADO Y PERSISTENCIA DE METADATOS DE TEST
# =====================================================================
print("\n--- Etapa 5: Inferencia por ensamble de fragmentos (Votación Avanzada) ---")

# Inicialización y unificación homogénea del diccionario de traducción estructural de categorías
estilos_dict = np.load('dict.npy', allow_pickle=True).item()
inverso_estilos_dict_corregido = {}
for k, v in estilos_dict.items():
    try:
        inverso_estilos_dict_corregido[int(v)] = k
    except ValueError:
        inverso_estilos_dict_corregido[v] = k

# Indexación sistemática de las señales digitales .mp3 no etiquetadas descargadas de Kaggle
archivos_audio_test = [f for f in os.listdir(ruta_audio_test_crudo) if f.endswith('.mp3')]
print(f"Total de archivos de audio identificados en el repositorio Kaggle: {len(archivos_audio_test)}")

nombres_archivos_salida = []
predicciones_finales_texto = []

for archivo_audio_name in archivos_audio_test:
    ruta_completa_audio = os.path.join(ruta_audio_test_crudo, archivo_audio_name)
    id_cancion = os.path.splitext(archivo_audio_name)[0]

    try:
        # Carga analítica de la señal acústica sin re-muestreo
        y, sr = librosa.load(ruta_completa_audio, sr=None)

        # Extracción analítica de los 13 coeficientes cepstrales originales
        mfcc_nativo = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=13)

        # PERSISTENCIA EN GOOGLE DRIVE: Salvaguarda física de las características de Test
        nombre_guardado_mfcc_drive = f"{id_cancion}_mfccs_test.npy"
        np.save(os.path.join(carpeta_mfccs_test_drive, nombre_guardado_mfcc_drive), mfcc_nativo)

        # Transposición matemática para consistencia en el eje temporal: (frames_totales, 13)
        mfcc_transpuesto = mfcc_nativo.T
        frames_totales_test = mfcc_transpuesto.shape[0]

        chunks_tokens_cancion = []

        # Segmentación temporal por ventanas deslizantes homólogas en memoria RAM (In-Memory Chunking)
        for start in range(0, frames_totales_test - CHUNK_FRAMES + 1, HOP_FRAMES):
            chunk_test = mfcc_transpuesto[start:start + CHUNK_FRAMES, :]

            # Cuantización utilizando la proyección del Codebook global unificado del entrenamiento
            tokens_test_chunk = kmeans_global.predict(chunk_test)
            chunks_tokens_cancion.append(tokens_test_chunk)

        if chunks_tokens_cancion:
            # PERSISTENCIA EN GOOGLE DRIVE: Salvaguarda física de las secuencias de tokens de Test
            tokens_lineales_cancion = np.concatenate(chunks_tokens_cancion)
            nombre_guardado_token_drive = f"{id_cancion}_mfccs_test_tokens_mfccs_test.npy"
            np.save(os.path.join(carpeta_tokens_test_drive, nombre_guardado_token_drive), tokens_lineales_cancion)

            # Homogeneización dimensional mediante la adición de padding pos-secuencia
            X_chunks_pred = pad_sequences(chunks_tokens_cancion, maxlen=MAX_LEN, padding='post', truncating='post')

            # Inferencia probabilística a través de la topología óptima guardada
            probabilidades_chunks = best_model_keras.predict(X_chunks_pred, verbose=0)
            clases_numpy = np.argmax(probabilidades_chunks, axis=1)

            # Criterio de agregación estadística mediante la moda empírica (Votación por Mayoría)
            clase_ganadora_np = np.bincount(clases_numpy).argmax()
            clase_ganadora_python = int(clase_ganadora_np) # Saneamiento de tipos np.int64 para evitar KeyErrors

            nombres_archivos_salida.append(archivo_audio_name)

            # Mapeo categórico de resguardo tolerante contra strings y enteros
            if clase_ganadora_python in inverso_estilos_dict_corregido:
                predicciones_finales_texto.append(inverso_estilos_dict_corregido[clase_ganadora_python])
            else:
                clase_str = str(clase_ganadora_python)
                predicciones_finales_texto.append(inverso_estilos_dict_corregido.get(clase_str, clase_str))

    except Exception as e:
        print(f"Advertencia: Omisión de la pista {archivo_audio_name} debido a fallos estructurales. Error: {e}")

# Consolidación del DataFrame final y exportación a almacenamiento local
submission = pd.DataFrame({'filename': nombres_archivos_salida, 'label': predicciones_finales_texto})
submission.to_csv('submission_kaggle_mir.csv', index=False)

print("\nPipeline completado de forma canónica. Archivo exportado para evaluación externa.")
print(submission.head(10))


--- Etapa 5: Inferencia por ensamble de fragmentos (Votación Avanzada) ---


NameError: name 'ruta_audio_test_crudo' is not defined